<a href="https://colab.research.google.com/github/franz-hufana/Thesis-MobileNet/blob/main/Optimized_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Thesis System: MobileNetV2 and MobileNetV3 Based Model

In [ ]:
import os
import gc
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from imageio.v2 import imread
from skimage.transform import resize
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2, MobileNetV3Large
from tensorflow.keras import backend as K
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from google.colab import files
from scipy.ndimage import label as cc_label
from matplotlib.patches import Rectangle

Import Necessary Datasets

In [ ]:
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
extract_path = "/content"

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
  zip_ref.extractall(extract_path)

print(f'Dataset extracted: {zip_name}')

Labelling the datasets according to their Classes and number of classes

In [ ]:
folder_path = '/content/woods'

label_map = {
    'c' : 0, # For Cracks
    'f' : 1, # For Fuzz
    'k' : 2, # For knotholes
    'n' : 3 # For No Defects
}

class_name = ['cracks', 'fuzz', 'knothole', 'no defect']
num_classes = 4
img_size = (224, 224) # Normalizing the images' size

print(f"Name of Clases {class_name}\nFolder Path: {folder_path}")

1. Preprocessing the Datasets to fit the MobileNet requirements

2. Dataset Shuffling

3. Splitting Train and Validation Set for our model

In [ ]:
print('Step 1: Data Preprocessing....')
print('Locating Datasets')
image_paths = []
for roots, dirs, files_in_dir in os.walk(folder_path):
  for file_name in file_in_dir:
    print('\nChecking Datasets')
    if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
      image_paths.append(os.ath.join(root, file_name))

print('\nSorting Datasets according to label')
image_paths = sorted(image_paths)

valid_data = []
valid_labels = []
valid_names = []

for file_path in image_paths:
  file_name os.path.basename(file_path)
  prefix = file_name[0].lower()

  if prefix not in label_map:
    print(f"Skiping unknown file: {file_name}")
    continue

  try:
    im = imread(file_path)

    if im.ndim == 2:
      print('\nConverted GrayScale Images to 3-channel (RGB)')
      im = np.stack([im] *3, axis=-1)

    elif im.ndim == 3 and im.shpe[-1] == 4:
      print('\nConverted RGBA Images to RGB')
      im = im[:, :, :3]

    print('\nResizing images to 224x224')
    im = resize(im, img_size, anti_aliasing = True, preserve_range = True)
    im = im.astype(np.float32)
    im = (im/127.5) - 1.0

    valid_data.append(im)
    valid_labels.append(label_map[prefix])
    valid_names.append(file_name)

  except Exception as e:
    print(f"Error reading {file_name}: {e}")


data = np.array(valid_data, dtype=np.float32)
labels = np.array(valid_labels, dtype=np.int32)

print(f'\nData Information (number of Datasets, Height, Width, Color Code): {data.shape}')
print(f'Label Information (number of Datasets): {labels.shape}')

print('\nSample Loaded files:')
for i in range(min(10, len(valid_names))):
  print(f"{valid_names[i]} --->> {class_name[labels[i]]}")


print('\nStep 2: Dataset Shuffling')
data, labels = shuffle(data, labels, random_state=42)
print('\nDataset Shuffled once only.')

print('\nStep 3: Splitting Dataset for Train and Validation....')

# For Validation
val_fraction = 0.2
val_size = int(len(data) * val_fraction)


x_val = data[:val_size]
y_val = labels[:val_size]

x_train = data[val_size:]
y_train = labels[val_size:]

print(f'\nTraining Set: {y_train.shape}\nValidation Set: {y_val.shape}')


Creating Layers for Data Augmentations

In [ ]:
def get_data_augmentation():
  return tf.keras.Sequential([
      layers.RandomFlip("horizontal"),
      layers.RandomRotation(0.22),
      layers.RandomZoom(0.2),
      layers.RandomContrast(0.2),
  ], name='data_augmentation')

Load the Models and Create New Classifier Head for Specific Task

In [ ]:
def build_model(backbone_name):
  inputs = layers.Input(Shape=(224, 224, 3), name="input_image")
  data_augmentation = get_data_augmentation()

  # Section for the Model
  if backbone_name == "MobileNetV2":
    base_model = MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet",
    )
  elif backbone_name == "MobileNetV3":
    base_model = MobileNetV3Large(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet",
        include_preprocessing=False
    )
  else:
    raise ValueError("Unsupported Model")

# Exclude already trained layers
base_model.trainable = False

# New Classifier Head for the Model
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = GlobalAveragePooling2D(name="gap")(x)
x = layers.Dropout(0.4, name="dropout")(x)
outputs = Dense(num_classes, activation = "softmax", name="wood_output")(x)

model = Model(inputs=inputs, outputs=outputs, name=f"{backbone_name}_wood_model")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

return model